# CLEVR VQA

Load a trained checkpoint from `projects/clevr/vqa/train.py` and inspect validation predictions.

In [ ]:
from pathlib import Path

import torch
from matplotlib import pyplot as plt

from chimera.data import CLEVRVQADataModule
from chimera.models import CLEVRVQAModel
from chimera.modules import VQAModule

data_dir = Path("/mnt/ai/data")
checkpoint_path = Path("/mnt/ai/runs/clevr/vqa/checkpoints/vqa.ckpt")

In [ ]:
dm = CLEVRVQADataModule(
    data_dir=data_dir, batch_size=8, num_workers=0, pin_memory=False
)
dm.prepare_data()
dm.setup("fit")

model = CLEVRVQAModel(vocab_size=dm.vocab_size, num_answers=dm.num_answers)
module = VQAModule.load_from_checkpoint(
    checkpoint_path,
    model=model,
    optimizer=None,
    scheduler=None,
    answer_names=dm.answer_names,
)
module.eval();

In [ ]:
batch = next(iter(dm.val_dataloader()))
with torch.no_grad():
    logits = module(batch["image"], batch["question"], batch["question_len"])
preds = logits.argmax(dim=1).tolist()

for i in range(min(4, len(preds))):
    image = batch["image"][i].permute(1, 2, 0).numpy()
    plt.figure(figsize=(3, 3))
    plt.imshow(image)
    plt.axis("off")
    plt.title(
        f"Q: {batch['question_text'][i]}\n"
        f"pred={dm.answer_names[preds[i]]} actual={batch['answer_text'][i]}",
        fontsize=8,
    )
    plt.show()